# Addestramento SFCN - Dataset IXI (T2) con Bias Correction Incorporata
Questo notebook contiene la pipeline di addestramento progettata per il dataset IXI. 

**Design**: Nessuna K-Fold. Eseguiamo un singolo Split Stratificato:
- **60% Training Set** (Addestramento Pesi)
- **20% Validation Set** (Early Stopping + Addestramento Retta di Regressione per Bias Correction)
- **20% Test Set** (Valutazione finale incontaminata)

**Novità**: Supporto configurabile per Transfer Learning (Full Fine-Tuning vs Partial Freezing). Permette di adattare i pesi ufficiali SFCN alla modalità T2 in tutta sicurezza. Aggiunto anche il supporto automatico per il caricamento dei file corrotti corretti.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('./SFCN')

In [ ]:
import os
import glob
import urllib.request
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datetime import datetime
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from dp_model.model_files.sfcn import SFCN
from train import train_model
from dp_model import dp_utils as dpu

PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Configurazione Parametri
Seleziona il tipo di Transfer Learning desiderato tramite `FREEZE_EARLY_LAYERS`.

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/collab4444/dataset-2-t1-t2/Prep_IXI_T2/Prep_IXI_T2"
CSV_PATH = "F:\\test\\ixi_info.csv"  # Aggiorna se esegui su Kaggle con dataset montato altrove!

# --- IMPOSTAZIONI FINE-TUNING ---
USE_FINETUNING = True
# Metti a False se stai passando da una modalità all'altra (es. Pretrain T1 -> Dataset T2), 
# in modo che i primissimi layer possano adattarsi al contrasto invertito! Metti a True per T1->T1.
FREEZE_EARLY_LAYERS = False  

PRETRAINED_URL = "https://raw.githubusercontent.com/ha-ha-ha-han/UKBiobank_deep_pretrain/master/brain_age/run_20190719_00_epoch_best_mae.p"
PRETRAINED_PATH = "/kaggle/working/sfcn_official_pretrained.prm"

OUTPUT_DIM = 100

EPOCHS = 130
BATCH_SIZE = 4
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 15

## 2. Definizione del Dataset IXI (con Gestione File Corrotti)

In [ ]:
class IXIBrainAgeDataset(Dataset):
    def __init__(self, data_dir, csv_path, is_train=False):
        self.data_dir = data_dir
        self.is_train = is_train
        self.samples = []
        
        self.bin_range = [0, OUTPUT_DIM]
        self.bin_step = 1
        self.sigma = 1.0
        
        df = pd.read_csv(csv_path)
        reference_date = datetime(2015, 2, 23)
        
        # Directory dove si trovano le versioni aggiustate dei file corrotti
        CORRECTED_DIR = "/kaggle/input/datasets/collab4444/ixi-t1-corrected-items/corrected_IXI_subjects_T2w"
        
        # Mappatura dalle sottocartelle del dataset principale a quelle dei file corretti
        subfolders_map = {
            'Prep_Guys_T2': 'Guys',
            'Prep_HH_T2': 'HH',
            'Prep_IOP_T2': 'IOP'
        }
        
        for sf, sf_corrected in subfolders_map.items():
            folder_path = os.path.join(data_dir, sf)
            if not os.path.exists(folder_path):
                continue
                
            for file in os.listdir(folder_path):
                if file.startswith("registered_image_") and file.endswith(".nii"):
                    nii_path = os.path.join(folder_path, file)
                    
                    # --- SOSTITUZIONE AUTOMATICA FILE CORROTTI ---
                    corrected_path = os.path.join(CORRECTED_DIR, sf_corrected, file)
                    if os.path.exists(corrected_path):
                        nii_path = corrected_path
                    
                    ixi_id_str = file.replace("registered_image_", "").replace(".nii", "")
                    try:
                        ixi_id = int(ixi_id_str)
                    except ValueError:
                        continue
                        
                    row = df[df['IXI_ID'] == ixi_id]
                    if len(row) == 0:
                        continue
                        
                    dob_str = row.iloc[0]['DOB']
                    if pd.isna(dob_str):
                        continue
                        
                    try:
                        dob_str = str(dob_str).split(" ")[0]
                        dob = datetime.strptime(dob_str, "%Y-%m-%d")
                        true_age = (reference_date - dob).days / 365.25
                        
                        y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
                        
                        self.samples.append({
                            "nii_path": nii_path,
                            "label_vect": y,
                            "true_age": true_age
                        })
                    except Exception as e:
                        continue
                        
        print(f"[{'TRAIN' if is_train else 'TEST/VAL'}] Caricati {len(self.samples)} pazienti IXI validi.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        if self.is_train:
            dx = np.random.randint(-3, 4)
            dy = np.random.randint(-3, 4)
            dz = np.random.randint(-3, 4)
        else:
            dx, dy, dz = 0, 0, 0
            
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        
        if self.is_train and np.random.rand() > 0.5:
            data = np.flip(data, axis=0).copy()
            
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_data, label_vect, sample['true_age']

## 3. Estrazione dei 3 Set Stratificati (60% Train, 20% Val, 20% Test)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

dummy_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=False)
dataset_size = len(dummy_dataset)

if dataset_size > 0:
    all_ages = [sample['true_age'] for sample in dummy_dataset.samples]
    all_indices = np.arange(dataset_size)
    
    strat_classes_all = [int(a/5) for a in all_ages]
    
    temp_idx, test_idx, temp_classes, _ = train_test_split(
        all_indices, strat_classes_all, 
        test_size=0.20, random_state=42, stratify=strat_classes_all
    )
    
    train_idx, val_idx = train_test_split(
        temp_idx, 
        test_size=0.25, random_state=42, stratify=temp_classes
    )
    
    print(f"\nSuddivisione completata:")
    print(f"- Pazienti Training (60%): {len(train_idx)}")
    print(f"- Pazienti Validation (20%): {len(val_idx)}")
    print(f"- Pazienti Test (20%): {len(test_idx)}")
    
    train_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=True)
    val_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=False)
    test_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=False)
    
    train_dataset = torch.utils.data.Subset(train_dataset, train_idx)
    val_dataset = torch.utils.data.Subset(val_dataset, val_idx)
    test_dataset = torch.utils.data.Subset(test_dataset, test_idx)
    
    train_ages = [all_ages[i] for i in train_idx]
    train_age_classes = [int(a/5) for a in train_ages]
    class_counts = np.bincount(train_age_classes)
    weights = [1.0 / class_counts[c] if class_counts[c] > 0 else 0 for c in train_age_classes]
    sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)
else:
    print("ERRORE: Impossibile trovare i file. Controlla i percorsi.")

## 4. Addestramento (con Transfer Learning)

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print(" INIZIO ADDESTRAMENTO RETE SFCN (IXI T2)")
    print("==============================================")
    
    model = SFCN(output_dim=OUTPUT_DIM)
    
    if USE_FINETUNING:
        if not os.path.exists(PRETRAINED_PATH):
            print("Scaricamento dei pesi ufficiali SFCN (UK Biobank) per il Transfer Learning...")
            urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)
            
        print("\nApplicazione pesi originali...")
        pretrained_state = torch.load(PRETRAINED_PATH, map_location='cpu')
        clean_state = {}
        for k, v in pretrained_state.items():
            name = k.replace("module.", "") if k.startswith("module.") else k
            clean_state[name] = v
            
        filtered_state = {k: v for k, v in clean_state.items() if not k.startswith('classifier.conv_6')}
        model.load_state_dict(filtered_state, strict=False)
        
        if FREEZE_EARLY_LAYERS:
            # Partial Freezing (Consigliato per T1 -> T1 su piccoli dataset)
            for name, param in model.named_parameters():
                if any(name.startswith(f"feature_extractor.conv_{i}") for i in range(4)):
                    param.requires_grad = False
            print("\u2714 Fine-Tuning: Primi 4 layer CONGELATI (Partial Freezing). Ultimi layer LIBERI.")
        else:
            # Full Fine-Tuning (Obbligatorio per T1 -> T2 per adattamento contrasto)
            print("\u2714 Fine-Tuning: TUTTI i layer sono LIBERI di adattarsi (Full Fine-Tuning).")
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = torch.optim.Adam(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = len(train_loader)
    
    trained_model, t_losses, v_losses, v_maes = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        epochs=EPOCHS,
        step_size=steps_per_epoch * 30,
        gamma=0.3,
        patience=PATIENCE
    )
    
    model_save_path = f"/kaggle/working/sfcn_IXI_T2_finetuned_model.pth"
    if isinstance(trained_model, nn.DataParallel):
        torch.save(trained_model.module.state_dict(), model_save_path)
    else:
        torch.save(trained_model.state_dict(), model_save_path)
        
    print(f"\nModello salvato in: {model_save_path}")

## 5. Age Bias Correction sul Validation Set & Test Finale

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print("   CALCOLO PREDIZIONI E BIAS CORRECTION")
    print("==============================================")
    
    trained_model.eval()
    bin_centers = np.arange(0, OUTPUT_DIM, 1)
    
    val_true_ages, val_preds = [], []
    with torch.no_grad():
        for inputs, _, true_age in val_loader:
            inputs = inputs.to(device)
            out = trained_model(inputs)[0].view(1, -1)
            prob = torch.exp(out).cpu().numpy()
            
            pred_age = (prob @ bin_centers)[0]
            
            val_true_ages.append(true_age.item())
            val_preds.append(pred_age)
            
    bias_model = LinearRegression()
    bias_model.fit(np.array(val_preds).reshape(-1, 1), np.array(val_true_ages))
    alpha = bias_model.coef_[0]
    beta = bias_model.intercept_
    print(f"Formula Bias Correction: Età_Corretta = {alpha:.3f} * Età_Predetta + {beta:.3f}")
    
    test_true_ages, test_preds = [], []
    with torch.no_grad():
        for inputs, _, true_age in test_loader:
            inputs = inputs.to(device)
            out = trained_model(inputs)[0].view(1, -1)
            prob = torch.exp(out).cpu().numpy()
            
            pred_age = (prob @ bin_centers)[0]
            
            test_true_ages.append(true_age.item())
            test_preds.append(pred_age)
            
    test_preds_corrected = bias_model.predict(np.array(test_preds).reshape(-1, 1))
    
    test_true_ages = np.array(test_true_ages)
    test_preds = np.array(test_preds)
    
    mae_pre = np.mean(np.abs(test_preds - test_true_ages))
    mae_post = np.mean(np.abs(test_preds_corrected - test_true_ages))
    
    print(f"\n>>> MAE SUL TEST SET (Pre-Correzione) : {mae_pre:.3f} Anni")
    print(f">>> MAE SUL TEST SET (Post-Correzione): {mae_post:.3f} Anni")
    
    plt.figure(figsize=(9, 7))
    plt.scatter(test_true_ages, test_preds, color='silver', edgecolor='gray', alpha=0.6, s=50, label='Pre-Correzione')
    plt.scatter(test_true_ages, test_preds_corrected, color='forestgreen', edgecolor='black', alpha=0.9, s=90, marker='*', label='Post-Correzione')
    
    min_val = min(min(test_true_ages), min(test_preds), min(test_preds_corrected)) - 2
    max_val = max(max(test_true_ages), max(test_preds), max(test_preds_corrected)) + 2
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2.5, label='Ideale')
    
    plt.title(f'Valutazione IXI (Fine-Tuning)\nMAE Finale: {mae_post:.3f} Anni\n(Correzione: Età = {alpha:.3f}*Pred + {beta:.3f})', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Età Reale (Anni)', fontsize=12)
    plt.ylabel('Età Predetta (Anni)', fontsize=12)
    plt.legend(loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, '09_ixi_finetuning_results.png'), dpi=300)
    plt.show()
